# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [21]:
from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [22]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [23]:
student_id = "student_03"
handle = "StareaNatiei"
max_videos = 1
max_comments_per_video = 50
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5\data\raw\student_03_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [26]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': '_Y0mOU5vGjIoINSwczj_UsAI9eE',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'IZfQ9-xARuCPmlf0AdHrbxa2gow',
   'id': 'UCtK5Oe8sHjp6WPcwWuHUVpQ'}]}

In [25]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCtK5Oe8sHjp6WPcwWuHUVpQ'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [27]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': '1k4-bCt2ImT9-KC8R_5ul5WaJfk',
 'id': {'kind': 'youtube#video', 'videoId': 'KbHoiGY4J4Q'},
 'snippet': {'publishedAt': '2026-05-16T07:00:23Z',
  'channelId': 'UCtK5Oe8sHjp6WPcwWuHUVpQ',
  'title': 'Workshop cu Gabriel Liiceanu și Dragoș Pătraru: Cum rămânem oameni într-o lume condusă de lichele',
  'description': 'Workshop cu Gabriel Liiceanu și Dragoș Pătraru: Cum rămânem oameni într-o lume condusă de lichele Bine-ați venit pe ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/KbHoiGY4J4Q/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/KbHoiGY4J4Q/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/KbHoiGY4J4Q/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Starea Natiei Oficial',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-16T07:00:23Z'}}

In [28]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'KbHoiGY4J4Q',
  'video_title': 'Workshop cu Gabriel Liiceanu și Dragoș Pătraru: Cum rămânem oameni într-o lume condusă de lichele',
  'video_date': '2026-05-16'}]

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [ ]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

In [42]:
# Create the raw directory if it doesn't exist
output_file.parent.mkdir(parents=True, exist_ok=True)

# Save the raw data before any cleaning happens
with output_file.open("w", encoding="utf-8") as f:
    for comment in comments:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii brute salvate:", len(comments))
print("Fișier raw:", output_file)

Comentarii brute salvate: 18
Fișier raw: c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5\data\raw\student_03_youtube_raw.jsonl


# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [30]:
comments[:3]

[{'id': 'yt_KbHoiGY4J4Q_UgyczFrKa4ldR3ZjUh14AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'StareaNatiei',
  'text_raw': 'Alt intelectual dreptac, să ne scutească',
  'video_id': 'KbHoiGY4J4Q',
  'video_title': 'Workshop cu Gabriel Liiceanu și Dragoș Pătraru: Cum rămânem oameni într-o lume condusă de lichele',
  'video_date': '2026-05-16',
  'comment_date': '2026-05-16',
  'likes': 2,
  'collected_at': '2026-05-16'},
 {'id': 'yt_KbHoiGY4J4Q_UgwjGrAqtbfteNaUnXh4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'StareaNatiei',
  'text_raw': 'În limba bătrână "să mă hăituiască" 😉.\n"Am glumit, dar am spus adevărul." Tare asta 👍😎.',
  'video_id': 'KbHoiGY4J4Q',
  'video_title': 'Workshop cu Gabriel Liiceanu și Dragoș Pătraru: Cum rămânem oameni într-o lume condusă de lichele',
  'video_date': '2026-05-16',
  'comment_date': '2026-05-16',
  'likes': 1,
  'collected_at': '2026-05-16'},
 {'id': 'yt_KbHoiGY4J4Q_UgzCTQmsIFjOvt_51TN4AaABAg',
  'source_platform': 'youtube

In [31]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [32]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [33]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_KbHoiGY4J4Q_UgyczFrKa4ldR3ZjUh14AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'StareaNatiei',
 'text_raw': 'Alt intelectual dreptac, să ne scutească',
 'video_id': 'KbHoiGY4J4Q',
 'video_title': 'Workshop cu Gabriel Liiceanu și Dragoș Pătraru: Cum rămânem oameni într-o lume condusă de lichele',
 'video_date': '2026-05-16',
 'comment_date': '2026-05-16',
 'likes': 2,
 'collected_at': '2026-05-16',
 'text': 'Alt intelectual dreptac, să ne scutească'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [34]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 18
Comentarii după filtrarea lungimii: 6


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [35]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 6


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [36]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 6


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [37]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 6
Fișier: c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5\data\cleaned\student_03_youtube_clean.jsonl


# Functia de curatare

In [38]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [39]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 18
Comentarii curate: 6


In [40]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: În limba bătrână "să mă hăituiască" 😉.
"Am glumit, dar am spus adevărul." Tare asta 👍😎.
CLEAN: În limba bătrână "să mă hăituiască" 😉. "Am glumit, dar am spus adevărul." Tare asta 👍😎.
---
RAW: Dacă considerați că Ilie Bolojan este MOISE, care trebuie clonat de 40 de ori, potrivit afirmațiilor lui Gabriel Liiceanu, atunci viitorul României este pecetluit!
CLEAN: Dacă considerați că Ilie Bolojan este MOISE, care trebuie clonat de 40 de ori, potrivit afirmațiilor lui Gabriel Liiceanu, atunci viitorul României este pecetluit!
---
RAW: 29:34 Am si eu impresia ca genul acesta de atitudine pe care o infatiseaza domnul Liceanu este nedemna si insultatoare. Ironic ca fix dansul vorbea mai devreme despre deschiderea pe care trebuie sa o aiba omul in lume, despre respectul in intampinarea unui altul.
CLEAN: 29:34 Am si eu impresia ca genul acesta de atitudine pe care o infatiseaza domnul Liceanu este nedemna si insultatoare. Ironic ca fix dansul vorbea mai devreme despre deschiderea pe care t

In [41]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5\data\cleaned\student_03_youtube_clean.jsonl
Comentarii salvate: 6


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.